In [ ]:
import socket
import time
import multiprocessing as mp

HOST = "192.168.2.99"
PORT = 50007

# ----------------------------
# BUZZER SQUARE WAVE (core requirement)
# ----------------------------
def write_buzzer_gpio(val: int):
    """
    TODO: Replace with your real GPIO write.
    Examples (depending on your lab setup):
      - pmoda_write(SIG_PIN, val)
      - gpio_write(pin, val)
    """
    # placeholder for localhost demo:
    pass

def play_tone(tone_freq_hz: float, duration_s: float = 0.5):
    """
    Implements the pseudocode from the assignment:
      write high
      sleep 1/(2*f)
      write low
      sleep 1/(2*f)
    :contentReference[oaicite:2]{index=2}
    """
    if tone_freq_hz <= 0:
        return
    half_period = 1.0 / (2.0 * tone_freq_hz)
    t_end = time.time() + duration_s

    while time.time() < t_end:
        write_buzzer_gpio(1)
        time.sleep(half_period)
        write_buzzer_gpio(0)
        time.sleep(half_period)

    write_buzzer_gpio(0)

# ----------------------------
# SERVER PROCESS
# ----------------------------
def server_proc(stop_evt: mp.Event):
    """
    Always running in listening mode :contentReference[oaicite:3]{index=3}
    Receives commands:
      - "TONE <freq> <dur>"
      - "QUIT"
    """
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind((HOST, PORT))
        s.listen(1)
        s.settimeout(0.25)  # lets us check stop_evt periodically

        print(f"[SERVER] Listening on {HOST}:{PORT}")

        conn = None
        try:
            while not stop_evt.is_set():
                if conn is None:
                    try:
                        conn, addr = s.accept()
                        conn.settimeout(0.25)
                        print(f"[SERVER] Client connected from {addr}")
                    except socket.timeout:
                        continue

                try:
                    data = conn.recv(1024)
                    if not data:
                        print("[SERVER] Client disconnected")
                        conn.close()
                        conn = None
                        continue

                    msg = data.decode("utf-8", errors="ignore").strip()
                    # print("[SERVER] RX:", msg)

                    if msg.startswith("TONE"):
                        # Format: TONE <freq> <dur>
                        parts = msg.split()
                        freq = float(parts[1]) if len(parts) >= 2 else 440.0
                        dur  = float(parts[2]) if len(parts) >= 3 else 0.5
                        print(f"[SERVER] Play tone: {freq} Hz for {dur} s")
                        play_tone(freq, dur)

                    elif msg == "QUIT":
                        print("[SERVER] QUIT received, shutting down")
                        stop_evt.set()
                        break

                except socket.timeout:
                    continue
                except Exception as e:
                    print("[SERVER] Error:", e)
                    if conn:
                        conn.close()
                        conn = None

        finally:
            if conn:
                conn.close()
            print("[SERVER] Stopped")

# ----------------------------
# CLIENT PROCESS
# ----------------------------
def client_proc(stop_evt: mp.Event):
    """
    Demo control loop.
    If you're on PYNQ, replace the keyboard input with button polling:
      - BTN0: connect
      - BTN1: send TONE
      - BTN2: disconnect (and end)
    """
    connected = False
    sock = None

    def connect():
        nonlocal connected, sock
        if connected:
            return
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.connect((HOST, PORT))
        connected = True
        print("[CLIENT] Connected")

    def send_tone(freq=440.0, dur=0.5):
        if not connected:
            print("[CLIENT] Not connected")
            return
        sock.sendall(f"TONE {freq} {dur}\n".encode("utf-8"))

    def disconnect_and_quit():
        nonlocal connected, sock
        if connected and sock:
            try:
                sock.sendall(b"QUIT\n")
            except Exception:
                pass
            try:
                sock.close()
            except Exception:
                pass
        connected = False
        sock = None
        stop_evt.set()
        print("[CLIENT] Disconnected + requested shutdown")

    print("[CLIENT] Controls: c=connect, t=tone, q=quit")
    while not stop_evt.is_set():
        # --- keyboard demo (swap this section with button polling on PYNQ) ---
        cmd = input("> ").strip().lower()
        if cmd == "c":
            connect()
        elif cmd == "t":
            send_tone(freq=880.0, dur=0.5)  # ~0.5s tone per press :contentReference[oaicite:4]{index=4}
        elif cmd == "q":
            disconnect_and_quit()
        else:
            print("Use: c, t, q")

    if sock:
        try: sock.close()
        except Exception: pass
    print("[CLIENT] Stopped")

# ----------------------------
# MAIN: start BOTH processes
# ----------------------------
if __name__ == "__main__":
    stop_evt = mp.Event()

    p_server = mp.Process(target=server_proc, args=(stop_evt,), name="server_proc")
    p_client = mp.Process(target=client_proc, args=(stop_evt,), name="client_proc")

    p_server.start()
    p_client.start()

    p_client.join()
    p_server.join()
    print("[MAIN] Done")